In [4]:
import yfinance as yf
import pandas as pd

# FOMC data covers 2011-2026; pull from 2010 for buffer (lets us compute
# returns cleanly at the left edge without a missing-prior-day gap)
spy = yf.download("SPY", start="2010-01-01", auto_adjust=True, progress=False)

# yfinance sometimes returns a MultiIndex column (('Close','SPY')) for single
# tickers depending on version — flatten it so downstream code is predictable
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

print("shape:", spy.shape)
print("columns:", list(spy.columns))
print("date range:", spy.index.min().date(), "->", spy.index.max().date())
print("nulls per column:")
print(spy.isnull().sum())
print("\nhead:")
print(spy.head(3))
print("\ntail:")
print(spy.tail(3))

shape: (4121, 5)
columns: ['Close', 'High', 'Low', 'Open', 'Volume']
date range: 2010-01-04 -> 2026-05-21
nulls per column:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

head:
Price           Close       High        Low       Open     Volume
Date                                                             
2010-01-04  84.796371  84.841263  83.434602  84.078076  118944600
2010-01-05  85.020836  85.058249  84.437221  84.743996  111579900
2010-01-06  85.080696  85.290198  84.871194  84.938531  116074400

tail:
Price            Close        High         Low        Open    Volume
Date                                                                
2026-05-19  733.729980  737.650024  731.530029  734.780029  54255900
2026-05-20  741.250000  741.869995  733.890015  735.710022  45768000
2026-05-21  742.719971  744.869995  737.030029  738.640015  43248300


In [5]:
# daily simple return: close[t] / close[t-1] - 1
# pct_change walks consecutive ROWS, and yfinance rows are trading days only,
# so this is automatically "previous trading day" — weekends/holidays skipped
spy["ret"] = spy["Close"].pct_change()

# target = NEXT trading day's return, aligned onto day t.
# shift(-1) pulls row t+1's value up to row t. Again, t+1 = next TRADING day,
# not calendar day, because the index has no non-trading rows.
spy["ret_next"] = spy["ret"].shift(-1)

# keep a clean frame: date + close + the two return cols
out = spy[["Close", "ret", "ret_next"]].copy()

print("shape:", out.shape)
print("\nhead:")
print(out.head(3))
print("\ntail (note last ret_next is NaN — no t+1 exists yet):")
print(out.tail(3))
print("\nnulls:")
print(out.isnull().sum())

shape: (4121, 3)

head:
Price           Close       ret  ret_next
Date                                     
2010-01-04  84.796371       NaN  0.002647
2010-01-05  85.020836  0.002647  0.000704
2010-01-06  85.080696  0.000704  0.004221

tail (note last ret_next is NaN — no t+1 exists yet):
Price            Close       ret  ret_next
Date                                      
2026-05-19  733.729980 -0.006661  0.010249
2026-05-20  741.250000  0.010249  0.001983
2026-05-21  742.719971  0.001983       NaN

nulls:
Price
Close       0
ret         1
ret_next    1
dtype: int64


In [6]:
import os

os.makedirs("../data/processed", exist_ok=True)

# index is the Date — keep it, it's our join key for aligning to FOMC dates later
out.to_csv("../data/processed/spy_daily.csv", index=True)

check = pd.read_csv("../data/processed/spy_daily.csv", index_col="Date", parse_dates=True)
print("saved rows:", len(check))
print("date range:", check.index.min().date(), "->", check.index.max().date())
print("columns:", list(check.columns))
print("\nhead:")
print(check.head(3))

saved rows: 4121
date range: 2010-01-04 -> 2026-05-21
columns: ['Close', 'ret', 'ret_next']

head:
                Close       ret  ret_next
Date                                     
2010-01-04  84.796371       NaN  0.002647
2010-01-05  85.020836  0.002647  0.000704
2010-01-06  85.080696  0.000704  0.004221


In [7]:
import os
print("cwd:", os.getcwd())
print("abs target:", os.path.abspath("data/processed/spy_daily.csv"))

cwd: /workspaces/fomc-sentiment-spy/notebooks
abs target: /workspaces/fomc-sentiment-spy/notebooks/data/processed/spy_daily.csv
